In [73]:
# https://www.kaggle.com/code/rhodiumbeng/classifying-multi-label-comments-0-9741-lb

In [74]:
import warnings
warnings.simplefilter("ignore")

In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [76]:
!pip install emoji

DEPRECATION: textract 1.6.5 has a non-standard dependency specifier extract-msg<=0.29.*. pip 23.3 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of textract or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [77]:
import string
import emoji

In [78]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load data

In [79]:
df = pd.read_csv('prachathai-67k.csv')

In [80]:
df.head(5)

,url,date,title,body_text,labels
0,https://prachatai.com/print/42,2004-08-24 14:31,"นักวิจัยหนุน ""แม้ว"" เปิด ""จีเอ็มโอ""",ประชาไท --- 23 ส.ค.2547 นักวิจัยฯ ชี้นโยบายจี...,"['ข่าว', 'สิ่งแวดล้อม']"
1,https://prachatai.com/print/41,2004-08-24 14:22,ภาคประชาชนต้านเปิดเสรีจีเอ็มโอ,ประชาไท- 23 ส.ค.2547 นักวิชาการ ภาคประชาชน จ...,"['ข่าว', 'สิ่งแวดล้อม']"
2,https://prachatai.com/print/43,2004-08-24 15:17,จุฬาฯ ห่วงจีเอ็มโอลามข้าวไทย,นโยบายที่อนุญาตให้ปลูกร่วมกับพืชอื่นได้นั้นถื...,"['ข่าว', 'สิ่งแวดล้อม']"
3,https://prachatai.com/print/45,2004-08-24 15:58,ฟองสบู่การเมืองแตก ทักษิณหมดกึ๋น ชนชั้นกลางหมด...,ประชาไท -- 23 ส.ค. 47 ขาประจำทักษิณ ฟันธง ฟอง...,"['ข่าว', 'การเมือง', 'คณะเศรษฐศาสตร์ มหาวิทยาล..."
4,https://prachatai.com/print/47,2004-08-24 16:10,กอต.เสนอเลิกถนนคลองลาน-อุ้มผาง,ประชาไท-23 ส.ค.47 คณะกรรมการอนุรักษ์ ผืนป่าตะ...,"['ข่าว', 'สิ่งแวดล้อม']"


In [81]:
df.shape

(67889, 5)

# 2. Check data quality

data type

In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67889 entries, 0 to 67888
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   url        67889 non-null  object
 1   date       67889 non-null  object
 2   title      67889 non-null  object
 3   body_text  67889 non-null  object
 4   labels     67889 non-null  object
dtypes: object(5)
memory usage: 2.6+ MB


unique value

In [83]:
for i in df.columns:
    print('Columns name: ', i)
    print('Unique value: ', df[i].unique())
    print('Count unique value: ', df[i].nunique())
    print('-'*10)

Columns name:  url
Unique value:  ['https://prachatai.com/print/42' 'https://prachatai.com/print/41'
 'https://prachatai.com/print/43' ... 'https://prachatai.com/print/79625'
 'https://prachatai.com/print/79628' 'https://prachatai.com/print/79629']
Count unique value:  67889
----------
Columns name:  date
Unique value:  ['2004-08-24 14:31' '2004-08-24 14:22' '2004-08-24 15:17' ...
 '2018-11-15 12:47' '2018-11-15 20:45' '2018-11-15 21:34']
Count unique value:  66732
----------
Columns name:  title
Unique value:  ['นักวิจัยหนุน  "แม้ว"  เปิด  "จีเอ็มโอ"' 'ภาคประชาชนต้านเปิดเสรีจีเอ็มโอ'
 'จุฬาฯ ห่วงจีเอ็มโอลามข้าวไทย' ...
 'ยุติวงจรผลัดกันเกาหลัง ด้วยการยื่นบัญชีทรัพย์สินฯ'
 'รัฐสวัสดิการ 101 กับ\xa0ภาคภูมิ แสงกนกกุล :\xa0การจัดวางความสัมพันธ์ใหม่ระหว่างรัฐ-สังคม'
 "ชาวอุยกูร์ในสหรัฐฯ ชุมนุมวันประกาศเอกราช เรียกร้องกดดันจีนกรณี 'ค่ายกักกัน'"]
Count unique value:  67417
----------
Columns name:  body_text
Unique value:  ['ประชาไท --- 23 ส.ค.2547  นักวิจัยฯ ชี้นโยบายจีเอ็มโอเอื้อต่อการค้นค

missing value

In [84]:
df.isna().sum()

url          0
date         0
title        0
body_text    0
labels       0
dtype: int64

pivot the label

In [85]:
benchmark_labels = ['การเมือง','สิทธิมนุษยชน','คุณภาพชีวิต','ต่างประเทศ','สังคม',
                  'สิ่งแวดล้อม','เศรษฐกิจ','วัฒนธรรม','แรงงาน','ความมั่นคง','ไอซีที','การศึกษา']

In [86]:
def expand_list_to_columns(row):
    classes = {c: int(c in row['labels']) for c in benchmark_labels}
    return pd.Series(classes)

In [87]:
new_cols = df.apply(expand_list_to_columns, axis=1)
df = pd.concat([df, new_cols], axis=1)

In [88]:
column_mapping = {
    "การเมือง": "politics",
    "สิทธิมนุษยชน": "human_rights",
    "คุณภาพชีวิต": "quality_of_life",
    "ต่างประเทศ": "foreign_affairs",
    "สังคม": "society",
    "สิ่งแวดล้อม": "environment",
    "เศรษฐกิจ": "economy",
    "วัฒนธรรม": "culture",
    "แรงงาน": "labor",
    "ความมั่นคง": "security",
    "ไอซีที": "ict",
    "การศึกษา": "education"
}

In [89]:
df = df.rename(columns=column_mapping)

In [90]:
df = df[['body_text','politics','human_rights','quality_of_life','foreign_affairs','society','environment','economy','culture','labor','security','ict','education']]

clean data

In [91]:
df.sample(10)

,body_text,politics,human_rights,quality_of_life,foreign_affairs,society,environment,economy,culture,labor,security,ict,education
66298,มติ สปสช. วางเป้าช่วยเหลือผู้ที่ตาบอด หรือมองเ...,0,0,1,0,0,0,0,0,0,0,0,0
39776,\n(4 มิ.ย.56) ที่ห้องประชุม 202 อาคารจามจุรี 4...,0,0,1,0,0,0,0,0,0,0,1,0
55269,22 เม.ย. 2559 จากกรณีที่วานนี้ (21 เม.ย.59) พล...,1,0,0,0,0,0,0,0,0,0,0,0
41092,บก. Wartani เปิดใจไม่กล้านอนบ้านหลังเหตุชายฉกร...,0,1,1,0,0,0,0,0,0,1,0,0
14118,\nขณะที่ค่ายเพลงยักษ์ใหญ่บางค่ายยังคงแสดงอาการ...,1,0,0,0,0,0,0,0,0,0,0,0
24570,อัดเละตีความเหมืองแร่ไม่เข้าข่ายโครงการพัฒนารุ...,0,1,1,0,0,1,0,0,0,0,0,0
9483,\nประชาไท - 22 ก.ย. 49 เมื่อวันที่ 21 ก.ย. นาย...,1,0,0,0,0,0,0,0,0,0,0,0
10567,\nทีมข่าวประชาไทภาคเหนือ : รายงาน\n\n \n\n\n\n...,0,0,0,0,0,0,0,0,0,0,0,0
15786,\nการเมือง\n\n \n\n \n\nสุรพงษ์หนุนแก้รธน. ฟื้...,1,0,0,0,0,0,0,0,0,0,0,0
8647,\nประชาไท8 ก.ค. 2549 นายสุริยะใส กตะศิลา ผู้ป...,1,0,0,0,0,0,0,0,0,0,0,0


In [105]:
import re
from pythainlp.util import normalize
from pythainlp import thai_characters
from pythainlp.corpus import thai_stopwords
from pythainlp.tokenize import word_tokenize

In [93]:
def normalize_text(text):
    return normalize(text)

In [94]:
def remove_html_tags(text):
    return re.sub(r'<[^>]+>', '', text)

In [95]:
def remove_urls(text):
    URL_PATTERN = r"""(?i)\b((?:https?:(?:/{1,3}|[a-z0-9%])|[a-z0-9.\-]+[.](?:com|net|org|edu|gov|mil|aero|asia|biz|cat|coop|info|int|jobs|mobi|museum|name|post|pro|tel|travel|xxx|ac|ad|ae|af|ag|ai|al|am|an|ao|aq|ar|as|at|au|aw|ax|az|ba|bb|bd|be|bf|bg|bh|bi|bj|bm|bn|bo|br|bs|bt|bv|bw|by|bz|ca|cc|cd|cf|cg|ch|ci|ck|cl|cm|cn|co|cr|cs|cu|cv|cx|cy|cz|dd|de|dj|dk|dm|do|dz|ec|ee|eg|eh|er|es|et|eu|fi|fj|fk|fm|fo|fr|ga|gb|gd|ge|gf|gg|gh|gi|gl|gm|gn|gp|gq|gr|gs|gt|gu|gw|gy|hk|hm|hn|hr|ht|hu|id|ie|il|im|in|io|iq|ir|is|it|je|jm|jo|jp|ke|kg|kh|ki|km|kn|kp|kr|kw|ky|kz|la|lb|lc|li|lk|lr|ls|lt|lu|lv|ly|ma|mc|md|me|mg|mh|mk|ml|mm|mn|mo|mp|mq|mr|ms|mt|mu|mv|mw|mx|my|mz|na|nc|ne|nf|ng|ni|nl|no|np|nr|nu|nz|om|pa|pe|pf|pg|ph|pk|pl|pm|pn|pr|ps|pt|pw|py|qa|re|ro|rs|ru|rw|sa|sb|sc|sd|se|sg|sh|si|sj|Ja|sk|sl|sm|sn|so|sr|ss|st|su|sv|sx|sy|sz|tc|td|tf|tg|th|tj|tk|tl|tm|tn|to|tp|tr|tt|tv|tw|tz|ua|ug|uk|us|uy|uz|va|vc|ve|vg|vi|vn|vu|wf|ws|ye|yt|yu|za|zm|zw)/)(?:[^\s()<>{}\[\]]+|\([^\s()]*?\([^\s()]+\)[^\s()]*?\)|\([^\s]+?\))+(?:\([^\s()]*?\([^\s()]+\)[^\s()]*?\)|\([^\s]+?\)|[^\s`!()\[\]{};:'".,<>?«»“”‘’])|(?:(?<!@)[a-z0-9]+(?:[.\-][a-z0-9]+)*[.](?:com|net|org|edu|gov|mil|aero|asia|biz|cat|coop|info|int|jobs|mobi|museum|name|post|pro|tel|travel|xxx|ac|ad|ae|af|ag|ai|al|am|an|ao|aq|ar|as|at|au|aw|ax|az|ba|bb|bd|be|bf|bg|bh|bi|bj|bm|bn|bo|br|bs|bt|bv|bw|by|bz|ca|cc|cd|cf|cg|ch|ci|ck|cl|cm|cn|co|cr|cs|cu|cv|cx|cy|cz|dd|de|dj|dk|dm|do|dz|ec|ee|eg|eh|er|es|et|eu|fi|fj|fk|fm|fo|fr|ga|gb|gd|ge|gf|gg|gh|gi|gl|gm|gn|gp|gq|gr|gs|gt|gu|gw|gy|hk|hm|hn|hr|ht|hu|id|ie|il|im|in|io|iq|ir|is|it|je|jm|jo|jp|ke|kg|kh|ki|km|kn|kp|kr|kw|ky|kz|la|lb|lc|li|lk|lr|ls|lt|lu|lv|ly|ma|mc|md|me|mg|mh|mk|ml|mm|mn|mo|mp|mq|mr|ms|mt|mu|mv|mw|mx|my|mz|na|nc|ne|nf|ng|ni|nl|no|np|nr|nu|nz|om|pa|pe|pf|pg|ph|pk|pl|pm|pn|pr|ps|pt|pw|py|qa|re|ro|rs|ru|rw|sa|sb|sc|sd|se|sg|sh|si|sj|Ja|sk|sl|sm|sn|so|sr|ss|st|su|sv|sx|sy|sz|tc|td|tf|tg|th|tj|tk|tl|tm|tn|to|tp|tr|tt|tv|tw|tz|ua|ug|uk|us|uy|uz|va|vc|ve|vg|vi|vn|vu|wf|ws|ye|yt|yu|za|zm|zw)\b/?(?!@)))"""
    return re.sub(URL_PATTERN, '', text)

In [96]:
def remove_thai_months(text):
    thai_months_full = ["มกราคม", "กุมภาพันธ์", "มีนาคม", "เมษายน", "พฤษภาคม", "มิถุนายน",
                        "กรกฎาคม", "สิงหาคม", "กันยายน", "ตุลาคม", "พฤศจิกายน", "ธันวาคม"]
    thai_months_abbr = ["ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.",
                        "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."]
    for month in thai_months_full + thai_months_abbr:
        text = text.replace(month, "")
    return text

In [97]:
def remove_non_thai_characters(text):
    allowed_characters = thai_characters + "ฯ"
    escaped_allowed_characters = re.escape(allowed_characters)
    pattern = '[^' + escaped_allowed_characters + ']'
    return re.sub(pattern, '', text)

In [98]:
def remove_stopwords(text):
    stopwords = thai_stopwords()
    text_tokens = text.split()
    text_tokens = [word for word in text_tokens if word not in stopwords]
    return ''.join(text_tokens)

In [100]:
def remove_special_characters(text):
    pattern = r'[!@#$%^&*()\-+=\[\]{};:\'",<.>/?\\|]\n'
    # Replace these special characters with an empty string.
    cleaned_text = re.sub(pattern, '', text)
    
    return cleaned_text

In [99]:
def remove_words_from_text(text):
    words_to_remove = ['ประชาไท']
    for word in words_to_remove:
        text = text.replace(word, '')
    return text

In [101]:
def remove_extra_spaces(text):
    return re.sub(r'\s+', '', text).strip()

In [102]:
df['body_text'] = df['body_text'].apply(normalize_text)
df['body_text'] = df['body_text'].apply(remove_html_tags)
df['body_text'] = df['body_text'].apply(remove_urls)
df['body_text'] = df['body_text'].apply(remove_thai_months)
df['body_text'] = df['body_text'].apply(remove_non_thai_characters)
df['body_text'] = df['body_text'].apply(remove_stopwords)
df['body_text'] = df['body_text'].apply(remove_special_characters)
df['body_text'] = df['body_text'].apply(remove_words_from_text)
df['body_text'] = df['body_text'].apply(remove_extra_spaces)

In [104]:
df.sample(20)

,body_text,politics,human_rights,quality_of_life,foreign_affairs,society,environment,economy,culture,labor,security,ict,education
57923,ภาพโดยเมื่อวันที่ที่ผ่านมาในการประชุมอนุสัญญาว...,0,0,0,1,0,1,0,0,0,0,0,0
26952,ที่ใดมีการกดขี่ที่นั่นย่อมมีการต่อต้านเป็นทฤษฎ...,1,0,0,0,0,0,0,0,0,0,0,0
26143,คณะทำงานยุติธรรมเพื่อสันติภาพออกแถลงการณ์ขอให้...,1,0,0,0,0,0,0,0,0,0,0,0
244,ศูนย์ข่าวภาคใต้สำนักงานตำรวจแห่งชาติสตชเปิดศูน...,0,0,0,0,0,1,0,0,0,0,0,0
214,เชียงใหม่ผศพาวินมะโนชัยรองคณบดีฝ่ายวิจัยและบริ...,0,0,0,0,0,1,0,0,0,0,0,0
4848,ชาวสะเมิงร่วมค้านโครงการก่อสร้างเขื่อนแม่ขานศู...,1,0,0,0,0,0,0,0,0,0,0,0
55067,ในยุครัฐบาลคสชการแสดงออกทางการเมืองถูกจำกัดมาก...,1,1,0,0,0,0,0,0,0,0,0,0
32752,ผู้นำเกาหลีเหนือให้ก่อสร้างส่วนจัดแสดงโลมาในเก...,0,0,1,1,0,0,0,0,0,0,0,0
2904,ปัตตานีเมื่อเวลานพตอประเสริฐจันทร์สว่างรองผู้บ...,0,0,0,0,0,1,0,0,0,0,0,0
32753,เป็นที่ถกเถียงกันว่าผู้ใช้โซเชียลเน็ตเวิร์กควร...,0,0,0,0,1,0,0,0,0,0,1,0


In [103]:
****

SyntaxError: invalid syntax (749935734.py, line 1)

In [106]:
def tokenize_text(text):
    return word_tokenize(text, engine='newmm')

In [107]:
df['body_text_tokenized'] = df['body_text'].apply(lambda x: tokenize_text(x))

KeyboardInterrupt: 

In [ ]:
df.sample(20)

# 3. Exploratory data analysis (EDA)

In [ ]:
target_col = ['obscene','insult','toxic','severe_toxic','identity_hate','threat']

In [ ]:
df[target_col].describe()

In [ ]:
for column in target_col:
    category_counts = df[column].value_counts()
    total_count = len(df[column])
    plt.figure(figsize=(6, 8))
    ax = category_counts.plot(kind='bar')
    for i, count in enumerate(category_counts):
        percentage = (count / total_count) * 100
        ax.annotate(f'{count} ({percentage:.2f}%)', xy=(i, count), ha='center', va='bottom')
    
    plt.xlabel(column)
    plt.ylabel('Count')
    plt.title(f'Count of {column}')
    plt.show()

In [ ]:
# All the class facing with imbalance problem.

In [ ]:
df['length'] = df['comment_text'].apply(lambda x: len(str(x)))

In [ ]:
plt.figure()
plt.hist(df['length'])
plt.show()

In [ ]:
# Some messages are over than 5000 words.

In [ ]:
target_corr = df[target_col].corr()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))
sns.heatmap(target_corr, annot=True, ax=ax)

In [ ]:
# Some of the labels are higher correlation such as obscene-insult/obscene-toxic/toxic-insult
# Classifier Chains would be good ?

# 4. Feature engineering

clean data

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "cannot ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)
    text = re.sub(r"\'scuse", " excuse ", text)
    text = re.sub('\W', ' ', text)
    text = re.sub('\s+', ' ', text)
    text = text.strip(' ')
    return text

In [ ]:
df['comment_text'] = df['comment_text'].map(lambda com : clean_text(com))

In [ ]:
df.sample(10)

# 5. Model

Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Splitting the dataset into the Training set and Test set
X_train, X_test, y_train, y_test = train_test_split(df[['comment_text']], df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]
                                                    ,test_size=0.2
                                                    ,random_state=42)

In [ ]:
print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

Vectorization/TD-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1,1))

In [ ]:
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf= TfidfVectorizer()

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
***

In [ ]:
def add_feature(X, feature_to_add):
    '''
    Returns sparse feature matrix with added feature.
    feature_to_add can also be a list of features.
    '''
    from scipy.sparse import csr_matrix, hstack
    return hstack([X, csr_matrix(feature_to_add).T], 'csr')

Model (Vectorization)

In [ ]:
# import and instantiate the Logistic Regression model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
logreg = LogisticRegression(C=12.0)

In [ ]:
for label in target_col:
    print('... Processing {}'.format(label))
    y = y_test[label]
    # train the model using X_dtm & y
    logreg.fit(X_train_tfidf, y)
    # compute the training accuracy
    y_pred_X = logreg.predict(X_test_tfidf)
    print('Training accuracy is {}'.format(accuracy_score(y, y_pred_X)))

In [ ]:
y

In [ ]:
X_train_tfidf